# 01 · Extraer consumos extremos respecto de la mediana de cada mercado · Resolución

Solución comentada de esta práctica de livecoding.

Seleccioná el kernel de `.venv` y usá **Reiniciar kernel y ejecutar todo**. La sección 3 contiene la función resuelta y la sección 4 ejecuta las pruebas de la práctica junto con la comprobación sobre datos reales. El resultado completo queda en la variable `resultado`.

La carga reutiliza `practicas_livecoding/preparacion.py` y los datasets ya descargados en `practicas_livecoding/datos/`. Conservá ambas carpetas dentro del mismo proyecto.

## 1. Preparación lista

En esta práctica se usa el equivalente a `df`: filas con ambas mediciones presentes.

Los archivos de `practicas_livecoding/datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_energia()
datos = datos.dropna(subset=["Consumo", "Consumidores"]).copy()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 271,811 | Columnas: 13


,TipoConsumidor,Sistema,UF,Classe,Setor Econômico - N2,Setor Econômico - N3,Tipo Tensão - N1,Tipo Tensão - N2,Tipo Tensão - N3,Faixa de Consumo N1,Faixa de Consumo N2,Consumidores,Consumo
Data,,,,,,,,,,,,,
2014-01-01,Cativo,Sudeste / Centro-Oeste,RO,Residencial,Convencional (Excepto Baixa Renda),TOTAL,A - Alta Tensão,TOTAL,TOTAL,Alta Tensão,Alta Tensão,1.0,8.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,SP,Rural,Serviço Público de Irrigação,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,6.0,145.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,SP,Rural,Serviço Público de Irrigação,TOTAL,B - Baixa Tensão,TOTAL,TOTAL,Não Aplicável,Não Aplicável,31.0,18.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,MS,Residencial,Baixa Renda,Baixa Renda N2,B - Baixa Tensão,TOTAL,TOTAL,Baixa Renda,> 200 kWh,44193.0,13171.0
2014-01-01,Cativo,Sistemas Isolados,AC,Residencial,Convencional (Excepto Baixa Renda),TOTAL,B - Baixa Tensão,TOTAL,TOTAL,Convencional,0-30 kWh,5216.0,149.0


## 2. Consigna y resultado esperado

Aplicá el ejercicio de las medianas al dataset de energía: por cada `TipoConsumidor`,
calculá la mediana de `Consumo` y extraé las `n` filas más alejadas en valor absoluto.

- La entrada ya contiene las filas con `Consumo` y `Consumidores` presentes.
  Conservá los consumos negativos y los ceros para este cálculo.
- Devolvé todas las columnas originales, seguidas por `mediana_grupo` y `distancia_mediana`.
- Ordená los mercados alfabéticamente y las distancias de mayor a menor en cada mercado.
  Los empates deben conservar el orden original de las filas.
- Conservá el índice `Data`, aunque esté repetido: distintas filas del mismo mes son válidas.
- `n` es positivo; los grupos menores que `n` aportan todas sus filas.
  Una entrada vacía devuelve una tabla vacía con las columnas esperadas.
- No modifiques la entrada ni elimines duplicados por fecha.

## 3. Resolución comentada

1. Calculamos y asignamos a cada fila la mediana de su `TipoConsumidor`.
2. Medimos la distancia absoluta del consumo a esa mediana, incluidos ceros y negativos.
3. Ordenamos de forma estable, tomamos `n` filas por mercado y ordenamos los mercados.

El índice `Data` puede repetirse porque un mes contiene varios segmentos.
No lo usamos como identificador único ni eliminamos filas por compartir fecha.

In [ ]:
def extraer_extremos_por_mercado(datos, n=5):
    """Extrae las n filas más alejadas de la mediana de consumo de cada mercado."""
    resultado = datos.copy()

    # La mediana se alinea con las filas, aunque el índice temporal se repita.
    resultado["mediana_grupo"] = resultado.groupby("TipoConsumidor")["Consumo"].transform("median")
    resultado["distancia_mediana"] = (resultado["Consumo"] - resultado["mediana_grupo"]).abs()

    # No filtramos ceros ni negativos: también participan en la comparación.
    ordenado = resultado.sort_values(
        "distancia_mediana", ascending=False, kind="stable"
    )
    extremos = ordenado.groupby("TipoConsumidor", sort=False).head(n)

    return extremos.sort_values("TipoConsumidor", kind="stable")

## 4. Pruebas y ejecución real

Se conservan los casos y las comprobaciones de la práctica original. Los mensajes `OK` confirman que pasan y que la entrada permanece intacta. La tabla muestra las primeras 20 filas; `resultado` contiene la salida completa.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "TipoConsumidor": ["B", "A", "A", "B", "A", "C", "B"],
        "Consumo": [10, 100, -20, 20, 40, 7, 30],
        "identificador": list("abcdefg"),
    }, index=pd.DatetimeIndex(["2023-01-01"] * 3 + ["2023-02-01"] * 4, name="Data"))
    original = caso.copy(deep=True)

    # Empates, grupos con menos de n filas y conservación de las columnas originales.
    esperado = caso.iloc[[1, 2, 0, 6, 5]].copy()
    esperado["mediana_grupo"] = [40, 40, 20, 20, 7]
    esperado["distancia_mediana"] = [60, 60, 10, 10, 0]
    comparar(funcion(caso, n=2), esperado)
    comparar(caso, original)

    comparar(funcion(caso, n=1), esperado.iloc[[0, 2, 4]])

    esperado_todos = caso.iloc[[1, 2, 4, 0, 6, 3, 5]].copy()
    esperado_todos["mediana_grupo"] = [40, 40, 40, 20, 20, 20, 7]
    esperado_todos["distancia_mediana"] = [60, 60, 0, 10, 10, 0, 0]
    comparar(funcion(caso, n=99), esperado_todos)

    vacio = caso.iloc[:0].copy()
    esperado_vacio = vacio.copy()
    esperado_vacio["mediana_grupo"] = pd.Series(dtype=float)
    esperado_vacio["distancia_mediana"] = pd.Series(dtype=float)
    comparar(funcion(vacio, n=5), esperado_vacio)
    comparar(caso, original)
    print("OK: extremos, empates, n variable, grupo pequeño, entrada vacía y datos intactos.")


probar(extraer_extremos_por_mercado)

# Comprobamos también que el dataset real permanezca intacto.
antes = datos.copy(deep=True)
resultado = extraer_extremos_por_mercado(datos)
comparar(datos, antes)
assert list(resultado.columns) == list(datos.columns) + ["mediana_grupo", "distancia_mediana"]
assert len(resultado) == datos.groupby("TipoConsumidor").size().clip(upper=5).sum()
for grupo, filas in resultado.groupby("TipoConsumidor"):
    valores = datos.loc[datos["TipoConsumidor"] == grupo, "Consumo"]
    distancias = np.sort(np.abs(valores.to_numpy() - np.median(valores)))[::-1][:5]
    np.testing.assert_allclose(filas["distancia_mediana"], distancias)
    np.testing.assert_allclose(filas["mediana_grupo"], np.median(valores))
print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
display(resultado.head(20))

OK: extremos, empates, n variable, grupo pequeño, entrada vacía y datos intactos.
OK: dataset real. Resultado: 10 filas.


,TipoConsumidor,Sistema,UF,Classe,Setor Econômico - N2,Setor Econômico - N3,Tipo Tensão - N1,Tipo Tensão - N2,Tipo Tensão - N3,Faixa de Consumo N1,Faixa de Consumo N2,Consumidores,Consumo,mediana_grupo,distancia_mediana
Data,,,,,,,,,,,,,,,
2014-11-01,Cativo,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,16723.000,1395433.000,1159.000,1394274.000
2014-10-01,Cativo,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,16725.000,1387076.000,1159.000,1385917.000
2014-02-01,Cativo,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,16756.000,1386881.000,1159.000,1385722.000
2014-03-01,Cativo,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,16746.000,1356867.000,1159.000,1355708.000
2014-09-01,Cativo,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,16724.000,1347298.000,1159.000,1346139.000
2022-04-01,Livre,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,A-2 - 88 a 138 kV,TOTAL,Não Aplicável,Não Aplicável,410.000,1850736.195,6017.614,1844718.581
2022-05-01,Livre,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,A-2 - 88 a 138 kV,TOTAL,Não Aplicável,Não Aplicável,413.000,1815405.730,6017.614,1809388.116
2022-08-01,Livre,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,A-2 - 88 a 138 kV,TOTAL,Não Aplicável,Não Aplicável,417.000,1813350.475,6017.614,1807332.861
2022-09-01,Livre,Sudeste / Centro-Oeste,SP,Industrial,TOTAL,TOTAL,A - Alta Tensão,A-2 - 88 a 138 kV,TOTAL,Não Aplicável,Não Aplicável,420.000,1805476.071,6017.614,1799458.457


## Para repasar

Volvé a la práctica sin resolver e intentá escribir la función desde cero. Explicá qué representa cada fila de la salida, cómo se resuelven los casos límite y por qué el cálculo respeta la consigna.